In [12]:
import sys
import subprocess

packages = [
    "pandas",
    "numpy",
    "scikit-learn",
    "joblib",
    "fastapi",
    "uvicorn[standard]",
    "pydantic",
    "pytest",
    "httpx",
    "streamlit",
    "requests"
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *packages])

print("Dependencias instaladas correctamente.")

Dependencias instaladas correctamente.


In [ ]:
from pathlib import Path

PROJECT_DIR = Path.home() / "bbva_mortgage_risk_project"

folders = [
    "app",
    "data",
    "models",
    "tests",
    "frontend"
]

for folder in folders:
    (PROJECT_DIR / folder).mkdir(parents=True, exist_ok=True)

(PROJECT_DIR / "app" / "__init__.py").write_text("", encoding="utf-8")

print("Proyecto creado en:")
print(PROJECT_DIR.resolve())

Proyecto creado en:
C:\Users\MI42678\bbva_mortgage_risk_project


In [14]:
from pathlib import Path

PROJECT_DIR = Path.home() / "bbva_mortgage_risk_project"

train_model_py = r'''
from pathlib import Path
import json
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

BASE_DIR = Path(__file__).resolve().parent
DATA_DIR = BASE_DIR / "data"
MODELS_DIR = BASE_DIR / "models"
DATA_DIR.mkdir(exist_ok=True)
MODELS_DIR.mkdir(exist_ok=True)


def calcular_pago_mensual(monto_credito, tasa_interes_anual, plazo_meses):
    tasa_mensual = (tasa_interes_anual / 100) / 12
    if np.any(tasa_mensual <= 0):
        return monto_credito / plazo_meses
    factor = (1 + tasa_mensual) ** plazo_meses
    return monto_credito * (tasa_mensual * factor) / (factor - 1)


def sigmoid(x):
    return 1 / (1 + np.exp(-x))


def generar_clientes_sinteticos(n=8000, seed=42):
    rng = np.random.default_rng(seed)

    edad = np.clip(rng.normal(39, 10, n).round(), 21, 75).astype(int)
    ingreso_mensual = np.clip(rng.lognormal(mean=np.log(42000), sigma=0.55, size=n), 9000, 260000).round(2)
    antiguedad_laboral_meses = np.clip(rng.gamma(shape=3.0, scale=24, size=n), 0, 420).round().astype(int)
    score_buro = np.clip(rng.normal(660, 75, n).round(), 430, 850).astype(int)

    atrasos_12m = rng.poisson(lam=np.where(score_buro < 600, 1.6, 0.45), size=n)
    atrasos_12m = np.clip(atrasos_12m, 0, 8).astype(int)

    cuentas_abiertas = np.clip(rng.poisson(5, n), 0, 20).astype(int)

    valor_vivienda = np.clip(
        rng.lognormal(mean=np.log(2200000), sigma=0.55, size=n),
        650000,
        12000000
    ).round(2)

    ltv = np.clip(rng.normal(0.73, 0.14, n), 0.35, 1.08)
    monto_credito = (valor_vivienda * ltv).round(2)

    plazo_meses = rng.choice([120, 180, 240, 300], size=n, p=[0.10, 0.25, 0.45, 0.20])
    tasa_interes_anual = np.clip(rng.normal(10.7, 1.35, n), 7.5, 15.5).round(2)

    deuda_mensual_actual = (
        ingreso_mensual * np.clip(rng.beta(2, 6, n), 0.02, 0.65)
    ).round(2)

    pago_mensual_estimado = calcular_pago_mensual(
        monto_credito,
        tasa_interes_anual,
        plazo_meses
    ).round(2)

    ratio_deuda_ingreso = (
        (deuda_mensual_actual + pago_mensual_estimado) / ingreso_mensual
    ).round(4)

    tipo_empleo = rng.choice(
        ["asalariado", "independiente", "pensionado", "informal"],
        size=n,
        p=[0.58, 0.24, 0.10, 0.08]
    )

    canal = rng.choice(
        ["sucursal", "digital", "broker"],
        size=n,
        p=[0.56, 0.30, 0.14]
    )

    zona = rng.choice(
        ["urbana", "suburbana", "rural"],
        size=n,
        p=[0.62, 0.27, 0.11]
    )

    efecto_empleo = np.select(
        [
            tipo_empleo == "asalariado",
            tipo_empleo == "independiente",
            tipo_empleo == "pensionado",
            tipo_empleo == "informal"
        ],
        [-0.15, 0.18, -0.05, 0.55],
        default=0
    )

    efecto_canal = np.select(
        [canal == "digital", canal == "broker"],
        [0.08, 0.14],
        default=0
    )

    efecto_zona = np.select(
        [zona == "rural", zona == "suburbana"],
        [0.12, 0.04],
        default=0
    )

    logit = (
        -2.20
        + 4.10 * (ltv - 0.72)
        + 3.40 * (ratio_deuda_ingreso - 0.45)
        + 0.018 * (635 - score_buro)
        + 0.33 * atrasos_12m
        - 0.0045 * antiguedad_laboral_meses
        + 0.0025 * (edad - 42)
        + efecto_empleo
        + efecto_canal
        + efecto_zona
        + rng.normal(0, 0.45, n)
    )

    prob_default = sigmoid(logit)
    default_12m = rng.binomial(1, prob_default)

    df = pd.DataFrame({
        "edad": edad,
        "ingreso_mensual": ingreso_mensual,
        "antiguedad_laboral_meses": antiguedad_laboral_meses,
        "score_buro": score_buro,
        "atrasos_12m": atrasos_12m,
        "cuentas_abiertas": cuentas_abiertas,
        "monto_credito": monto_credito,
        "valor_vivienda": valor_vivienda,
        "plazo_meses": plazo_meses,
        "tasa_interes_anual": tasa_interes_anual,
        "deuda_mensual_actual": deuda_mensual_actual,
        "ltv": ltv.round(4),
        "pago_mensual_estimado": pago_mensual_estimado,
        "ratio_deuda_ingreso": ratio_deuda_ingreso,
        "tipo_empleo": tipo_empleo,
        "canal": canal,
        "zona": zona,
        "default_12m": default_12m
    })

    return df


def crear_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def entrenar_modelo():
    df = generar_clientes_sinteticos()

    data_path = DATA_DIR / "clientes_hipotecarios.csv"
    df.to_csv(data_path, index=False, encoding="utf-8")

    target = "default_12m"

    features = [
        "edad",
        "ingreso_mensual",
        "antiguedad_laboral_meses",
        "score_buro",
        "atrasos_12m",
        "cuentas_abiertas",
        "monto_credito",
        "valor_vivienda",
        "plazo_meses",
        "tasa_interes_anual",
        "deuda_mensual_actual",
        "ltv",
        "pago_mensual_estimado",
        "ratio_deuda_ingreso",
        "tipo_empleo",
        "canal",
        "zona"
    ]

    categorical_features = ["tipo_empleo", "canal", "zona"]
    numeric_features = [col for col in features if col not in categorical_features]

    X = df[features]
    y = df[target]

    X_train, X_test, y_train, y_test = train_test_split(
        X,
        y,
        test_size=0.25,
        random_state=42,
        stratify=y
    )

    preprocessor = ColumnTransformer([
        ("num", StandardScaler(), numeric_features),
        ("cat", crear_encoder(), categorical_features)
    ])

    model = RandomForestClassifier(
        n_estimators=260,
        max_depth=9,
        min_samples_leaf=18,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    pipeline = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    pipeline.fit(X_train, y_train)

    y_pred = pipeline.predict(X_test)
    y_prob = pipeline.predict_proba(X_test)[:, 1]

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    metrics = {
        "accuracy": round(float(accuracy_score(y_test, y_pred)), 4),
        "precision": round(float(precision_score(y_test, y_pred, zero_division=0)), 4),
        "recall": round(float(recall_score(y_test, y_pred, zero_division=0)), 4),
        "f1": round(float(f1_score(y_test, y_pred, zero_division=0)), 4),
        "roc_auc": round(float(roc_auc_score(y_test, y_prob)), 4),
        "default_rate_dataset": round(float(y.mean()), 4),
        "confusion_matrix": {
            "tn": int(tn),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp)
        }
    }

    model_path = MODELS_DIR / "modelo_riesgo_hipotecario.pkl"
    metadata_path = MODELS_DIR / "modelo_metadata.json"

    joblib.dump(pipeline, model_path)

    metadata = {
        "project_name": "BBVA Mortgage Risk API - Proyecto academico",
        "model_file": model_path.name,
        "data_file": data_path.name,
        "model_type": "RandomForestClassifier con preprocesamiento",
        "target": target,
        "features": features,
        "numeric_features": numeric_features,
        "categorical_features": categorical_features,
        "n_rows": int(len(df)),
        "created_at": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
        "metrics": metrics,
        "risk_thresholds": {
            "bajo": "< 0.25",
            "medio": "0.25 - 0.4499",
            "alto": "0.45 - 0.6499",
            "critico": ">= 0.65"
        },
        "important_note": "Datos 100% sinteticos. No usar para decisiones reales de credito."
    }

    metadata_path.write_text(
        json.dumps(metadata, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )

    return metadata


if __name__ == "__main__":
    metadata = entrenar_modelo()
    print("Modelo entrenado correctamente.")
    print(json.dumps(metadata["metrics"], indent=2, ensure_ascii=False))
'''

(PROJECT_DIR / "train_model.py").write_text(train_model_py, encoding="utf-8")

print("Archivo train_model.py creado correctamente.")
print(PROJECT_DIR / "train_model.py")

Archivo train_model.py creado correctamente.
C:\Users\MI42678\bbva_mortgage_risk_project\train_model.py


In [15]:
from pathlib import Path

PROJECT_DIR = Path.home() / "bbva_mortgage_risk_project"

schemas_py = r'''
from typing import List, Literal
from pydantic import BaseModel, Field

TipoEmpleo = Literal["asalariado", "independiente", "pensionado", "informal"]
Canal = Literal["sucursal", "digital", "broker"]
Zona = Literal["urbana", "suburbana", "rural"]


class ClienteHipotecarioInput(BaseModel):
    edad: int = Field(..., ge=18, le=80)
    ingreso_mensual: float = Field(..., gt=0)
    antiguedad_laboral_meses: int = Field(..., ge=0, le=600)
    score_buro: int = Field(..., ge=300, le=900)
    atrasos_12m: int = Field(..., ge=0, le=20)
    cuentas_abiertas: int = Field(..., ge=0, le=50)
    monto_credito: float = Field(..., gt=0)
    valor_vivienda: float = Field(..., gt=0)
    plazo_meses: int = Field(..., ge=60, le=360)
    tasa_interes_anual: float = Field(..., gt=0, le=30)
    deuda_mensual_actual: float = Field(..., ge=0)
    tipo_empleo: TipoEmpleo
    canal: Canal
    zona: Zona


class PrediccionResponse(BaseModel):
    probabilidad_incumplimiento: float
    nivel_riesgo: str
    decision_sugerida: str
    pago_mensual_estimado: float
    ltv: float
    ratio_deuda_ingreso: float
    factores_riesgo: List[str]
    factores_mitigantes: List[str]
    nota: str


class BatchPredictionInput(BaseModel):
    clientes: List[ClienteHipotecarioInput]


class BatchPredictionResponse(BaseModel):
    total_clientes: int
    resultados: List[PrediccionResponse]
'''

risk_rules_py = r'''
from typing import Any, Dict, List, Tuple


def calcular_pago_mensual(monto_credito: float, tasa_interes_anual: float, plazo_meses: int) -> float:
    tasa_mensual = (tasa_interes_anual / 100) / 12

    if tasa_mensual <= 0:
        return monto_credito / plazo_meses

    factor = (1 + tasa_mensual) ** plazo_meses
    return monto_credito * (tasa_mensual * factor) / (factor - 1)


def normalizar_payload(payload: Any) -> Dict[str, Any]:
    if hasattr(payload, "model_dump"):
        return payload.model_dump()

    if hasattr(payload, "dict"):
        return payload.dict()

    return dict(payload)


def enriquecer_variables(payload: Any) -> Dict[str, Any]:
    data = normalizar_payload(payload)

    valor_vivienda = max(float(data["valor_vivienda"]), 1.0)
    ingreso_mensual = max(float(data["ingreso_mensual"]), 1.0)

    pago = calcular_pago_mensual(
        monto_credito=float(data["monto_credito"]),
        tasa_interes_anual=float(data["tasa_interes_anual"]),
        plazo_meses=int(data["plazo_meses"])
    )

    ltv = float(data["monto_credito"]) / valor_vivienda
    ratio_deuda_ingreso = (float(data["deuda_mensual_actual"]) + pago) / ingreso_mensual

    enriched = dict(data)
    enriched["ltv"] = round(ltv, 4)
    enriched["pago_mensual_estimado"] = round(pago, 2)
    enriched["ratio_deuda_ingreso"] = round(ratio_deuda_ingreso, 4)

    return enriched


def clasificar_riesgo(probabilidad: float) -> str:
    if probabilidad < 0.25:
        return "Riesgo bajo"

    if probabilidad < 0.45:
        return "Riesgo medio"

    if probabilidad < 0.65:
        return "Riesgo alto"

    return "Riesgo critico"


def decision_sugerida(nivel_riesgo: str) -> str:
    decisiones = {
        "Riesgo bajo": "Preaprobar sujeto a validacion documental.",
        "Riesgo medio": "Solicitar revision adicional y validar capacidad de pago.",
        "Riesgo alto": "Requiere analisis manual, posibles ajustes de monto, enganche o plazo.",
        "Riesgo critico": "No aprobar en condiciones actuales; solicitar mitigantes o reestructura de la propuesta."
    }

    return decisiones.get(nivel_riesgo, "Revisar caso manualmente.")


def explicar_prediccion(features: Dict[str, Any]) -> Tuple[List[str], List[str]]:
    factores_riesgo = []
    factores_mitigantes = []

    if features["ltv"] >= 0.90:
        factores_riesgo.append("LTV elevado: el monto del credito representa una proporcion alta del valor de la vivienda.")
    elif features["ltv"] <= 0.70:
        factores_mitigantes.append("LTV saludable: existe mayor aportacion inicial o menor exposicion relativa.")

    if features["ratio_deuda_ingreso"] >= 0.55:
        factores_riesgo.append("Alta relacion deuda/ingreso: la carga mensual estimada supera niveles prudentes.")
    elif features["ratio_deuda_ingreso"] <= 0.35:
        factores_mitigantes.append("Relacion deuda/ingreso conservadora.")

    if features["score_buro"] < 600:
        factores_riesgo.append("Score de buro bajo frente al rango esperado.")
    elif features["score_buro"] >= 720:
        factores_mitigantes.append("Score de buro alto.")

    if features["atrasos_12m"] >= 3:
        factores_riesgo.append("Historial reciente con multiples atrasos.")
    elif features["atrasos_12m"] == 0:
        factores_mitigantes.append("Sin atrasos registrados en los ultimos 12 meses.")

    if features["antiguedad_laboral_meses"] < 12:
        factores_riesgo.append("Antiguedad laboral menor a 12 meses.")
    elif features["antiguedad_laboral_meses"] >= 48:
        factores_mitigantes.append("Antiguedad laboral estable.")

    if features["tipo_empleo"] == "informal":
        factores_riesgo.append("Tipo de empleo informal: mayor incertidumbre en comprobacion de ingresos.")
    elif features["tipo_empleo"] == "asalariado":
        factores_mitigantes.append("Ingreso asalariado: perfil con mayor estabilidad documental.")

    if features["pago_mensual_estimado"] > features["ingreso_mensual"] * 0.45:
        factores_riesgo.append("Pago hipotecario estimado alto respecto al ingreso mensual.")

    if not factores_riesgo:
        factores_riesgo.append("No se identificaron alertas criticas con las reglas explicativas.")

    if not factores_mitigantes:
        factores_mitigantes.append("No se identificaron mitigantes fuertes con las reglas explicativas.")

    return factores_riesgo[:5], factores_mitigantes[:5]
'''

model_utils_py = r'''
import json
from functools import lru_cache
from pathlib import Path
from typing import Any, Dict, Tuple

import joblib
import pandas as pd

from .risk_rules import clasificar_riesgo, decision_sugerida, enriquecer_variables, explicar_prediccion

BASE_DIR = Path(__file__).resolve().parents[1]
MODEL_PATH = BASE_DIR / "models" / "modelo_riesgo_hipotecario.pkl"
METADATA_PATH = BASE_DIR / "models" / "modelo_metadata.json"


@lru_cache(maxsize=1)
def cargar_modelo() -> Tuple[Any, Dict[str, Any]]:
    if not MODEL_PATH.exists():
        raise FileNotFoundError(
            f"No existe el modelo en {MODEL_PATH}. Ejecuta primero: python train_model.py"
        )

    if not METADATA_PATH.exists():
        raise FileNotFoundError(
            f"No existe la metadata en {METADATA_PATH}. Ejecuta primero: python train_model.py"
        )

    model = joblib.load(MODEL_PATH)
    metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))

    return model, metadata


def predecir_riesgo(payload: Any) -> Dict[str, Any]:
    model, metadata = cargar_modelo()

    features = enriquecer_variables(payload)

    X = pd.DataFrame([features])
    X = X[metadata["features"]]

    probabilidad = float(model.predict_proba(X)[0, 1])
    nivel = clasificar_riesgo(probabilidad)

    factores_riesgo, factores_mitigantes = explicar_prediccion(features)

    return {
        "probabilidad_incumplimiento": round(probabilidad, 4),
        "nivel_riesgo": nivel,
        "decision_sugerida": decision_sugerida(nivel),
        "pago_mensual_estimado": round(float(features["pago_mensual_estimado"]), 2),
        "ltv": round(float(features["ltv"]), 4),
        "ratio_deuda_ingreso": round(float(features["ratio_deuda_ingreso"]), 4),
        "factores_riesgo": factores_riesgo,
        "factores_mitigantes": factores_mitigantes,
        "nota": "Resultado generado con datos ficticios para un proyecto academico. No usar para decisiones reales de credito."
    }


def obtener_info_modelo() -> Dict[str, Any]:
    _, metadata = cargar_modelo()
    return metadata
'''

main_py = r'''
from contextlib import asynccontextmanager

from fastapi import FastAPI

from .model_utils import cargar_modelo, obtener_info_modelo, predecir_riesgo
from .schemas import BatchPredictionInput, BatchPredictionResponse, ClienteHipotecarioInput, PrediccionResponse


@asynccontextmanager
async def lifespan(app: FastAPI):
    cargar_modelo()
    yield


app = FastAPI(
    title="API de Evaluacion de Riesgo Hipotecario",
    description="Proyecto academico: estimacion de riesgo de incumplimiento para clientes hipotecarios ficticios.",
    version="1.0.0",
    lifespan=lifespan
)


@app.get("/")
def root() -> dict:
    return {
        "mensaje": "API de riesgo hipotecario ejecutandose correctamente",
        "docs": "/docs",
        "health": "/health"
    }


@app.get("/health")
def health() -> dict:
    cargar_modelo()
    return {
        "status": "ok",
        "model": "loaded"
    }


@app.get("/model-info")
def model_info() -> dict:
    return obtener_info_modelo()


@app.post("/predict", response_model=PrediccionResponse)
def predict(cliente: ClienteHipotecarioInput) -> dict:
    return predecir_riesgo(cliente)


@app.post("/predict-batch", response_model=BatchPredictionResponse)
def predict_batch(batch: BatchPredictionInput) -> dict:
    resultados = [predecir_riesgo(cliente) for cliente in batch.clientes]

    return {
        "total_clientes": len(resultados),
        "resultados": resultados
    }
'''

(PROJECT_DIR / "app" / "schemas.py").write_text(schemas_py, encoding="utf-8")
(PROJECT_DIR / "app" / "risk_rules.py").write_text(risk_rules_py, encoding="utf-8")
(PROJECT_DIR / "app" / "model_utils.py").write_text(model_utils_py, encoding="utf-8")
(PROJECT_DIR / "app" / "main.py").write_text(main_py, encoding="utf-8")

print("Archivos del backend API creados correctamente.")

Archivos del backend API creados correctamente.


In [16]:
from pathlib import Path

PROJECT_DIR = Path.home() / "bbva_mortgage_risk_project"

frontend_py = r'''
import os
import requests
import streamlit as st

API_URL = os.getenv("API_URL", "http://127.0.0.1:8000")

st.set_page_config(
    page_title="Evaluacion de Riesgo Hipotecario",
    layout="centered"
)

st.title("Evaluacion de Riesgo Hipotecario")
st.write("Frontend academico conectado a una API FastAPI. Datos ficticios, no uso productivo.")

with st.form("formulario_riesgo"):
    st.subheader("Datos del cliente")

    edad = st.number_input("Edad", min_value=18, max_value=80, value=36)
    ingreso_mensual = st.number_input("Ingreso mensual", min_value=1.0, value=62000.0, step=1000.0)
    antiguedad_laboral_meses = st.number_input("Antiguedad laboral en meses", min_value=0, max_value=600, value=72)
    score_buro = st.number_input("Score buro ficticio", min_value=300, max_value=900, value=735)
    atrasos_12m = st.number_input("Atrasos en ultimos 12 meses", min_value=0, max_value=20, value=0)
    cuentas_abiertas = st.number_input("Cuentas abiertas", min_value=0, max_value=50, value=4)

    st.subheader("Datos del credito")

    monto_credito = st.number_input("Monto del credito", min_value=1.0, value=1800000.0, step=50000.0)
    valor_vivienda = st.number_input("Valor de la vivienda", min_value=1.0, value=2800000.0, step=50000.0)
    plazo_meses = st.selectbox("Plazo en meses", [120, 180, 240, 300], index=2)
    tasa_interes_anual = st.number_input("Tasa de interes anual (%)", min_value=0.1, max_value=30.0, value=10.2, step=0.1)
    deuda_mensual_actual = st.number_input("Deuda mensual actual", min_value=0.0, value=4500.0, step=500.0)

    st.subheader("Perfil")

    tipo_empleo = st.selectbox("Tipo de empleo", ["asalariado", "independiente", "pensionado", "informal"])
    canal = st.selectbox("Canal", ["sucursal", "digital", "broker"])
    zona = st.selectbox("Zona", ["urbana", "suburbana", "rural"])

    submitted = st.form_submit_button("Evaluar riesgo")

if submitted:
    payload = {
        "edad": int(edad),
        "ingreso_mensual": float(ingreso_mensual),
        "antiguedad_laboral_meses": int(antiguedad_laboral_meses),
        "score_buro": int(score_buro),
        "atrasos_12m": int(atrasos_12m),
        "cuentas_abiertas": int(cuentas_abiertas),
        "monto_credito": float(monto_credito),
        "valor_vivienda": float(valor_vivienda),
        "plazo_meses": int(plazo_meses),
        "tasa_interes_anual": float(tasa_interes_anual),
        "deuda_mensual_actual": float(deuda_mensual_actual),
        "tipo_empleo": tipo_empleo,
        "canal": canal,
        "zona": zona
    }

    try:
        response = requests.post(
            f"{API_URL}/predict",
            json=payload,
            timeout=15
        )

        response.raise_for_status()
        result = response.json()

        st.success(result["nivel_riesgo"])
        st.metric("Probabilidad de incumplimiento", f"{result['probabilidad_incumplimiento']:.2%}")
        st.metric("LTV", f"{result['ltv']:.2%}")
        st.metric("Ratio deuda/ingreso", f"{result['ratio_deuda_ingreso']:.2%}")

        st.write("Decision sugerida:")
        st.info(result["decision_sugerida"])

        st.write("Factores de riesgo")
        for item in result["factores_riesgo"]:
            st.write(f"- {item}")

        st.write("Factores mitigantes")
        for item in result["factores_mitigantes"]:
            st.write(f"- {item}")

    except requests.exceptions.RequestException as exc:
        st.error(f"No fue posible conectar con la API en {API_URL}. Detalle: {exc}")
'''

test_api_py = r'''
from fastapi.testclient import TestClient

from app.main import app


client = TestClient(app)


BASE_PAYLOAD = {
    "edad": 36,
    "ingreso_mensual": 62000,
    "antiguedad_laboral_meses": 72,
    "score_buro": 735,
    "atrasos_12m": 0,
    "cuentas_abiertas": 4,
    "monto_credito": 1800000,
    "valor_vivienda": 2800000,
    "plazo_meses": 240,
    "tasa_interes_anual": 10.2,
    "deuda_mensual_actual": 4500,
    "tipo_empleo": "asalariado",
    "canal": "sucursal",
    "zona": "urbana"
}


def test_health():
    response = client.get("/health")

    assert response.status_code == 200
    assert response.json()["status"] == "ok"


def test_predict_success():
    response = client.post("/predict", json=BASE_PAYLOAD)

    assert response.status_code == 200

    data = response.json()

    assert 0 <= data["probabilidad_incumplimiento"] <= 1
    assert data["nivel_riesgo"] in [
        "Riesgo bajo",
        "Riesgo medio",
        "Riesgo alto",
        "Riesgo critico"
    ]
    assert data["ltv"] > 0
    assert data["ratio_deuda_ingreso"] > 0


def test_predict_high_risk_case():
    payload = dict(BASE_PAYLOAD)

    payload.update({
        "ingreso_mensual": 25000,
        "score_buro": 540,
        "atrasos_12m": 5,
        "monto_credito": 2700000,
        "valor_vivienda": 2800000,
        "antiguedad_laboral_meses": 5,
        "tipo_empleo": "informal"
    })

    response = client.post("/predict", json=payload)

    assert response.status_code == 200

    data = response.json()

    assert data["probabilidad_incumplimiento"] >= 0
    assert len(data["factores_riesgo"]) > 0


def test_predict_validation_error():
    payload = dict(BASE_PAYLOAD)
    payload["edad"] = 15

    response = client.post("/predict", json=payload)

    assert response.status_code == 422
'''

requirements_txt = '''
fastapi>=0.110.0
uvicorn[standard]>=0.27.0
pydantic>=2.0.0
pandas>=2.0.0
numpy>=1.24.0
scikit-learn>=1.3.0
joblib>=1.3.0
pytest>=8.0.0
httpx>=0.25.0
streamlit>=1.32.0
requests>=2.31.0
'''.strip() + "\n"

dockerfile_api = '''
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY . .
RUN python train_model.py

EXPOSE 8000

CMD ["uvicorn", "app.main:app", "--host", "0.0.0.0", "--port", "8000"]
'''.strip() + "\n"

dockerfile_frontend = '''
FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

COPY frontend ./frontend

EXPOSE 8501

CMD ["streamlit", "run", "frontend/streamlit_app.py", "--server.address", "0.0.0.0", "--server.port", "8501"]
'''.strip() + "\n"

docker_compose = '''
services:
  api:
    build:
      context: .
      dockerfile: Dockerfile
    container_name: bbva-mortgage-risk-api
    ports:
      - "8000:8000"
    environment:
      - PYTHONUNBUFFERED=1

  frontend:
    build:
      context: .
      dockerfile: Dockerfile.frontend
    container_name: bbva-mortgage-risk-frontend
    ports:
      - "8501:8501"
    environment:
      - API_URL=http://api:8000
    depends_on:
      - api
'''.strip() + "\n"

dockerignore = '''
__pycache__/
*.pyc
.ipynb_checkpoints/
.venv/
venv/
.env
.git/
.pytest_cache/
'''.strip() + "\n"

gitignore = '''
__pycache__/
*.pyc
.ipynb_checkpoints/
.venv/
venv/
.env
.pytest_cache/
.DS_Store
'''.strip() + "\n"

readme_md = r'''
# API de Evaluacion de Riesgo Hipotecario

Proyecto academico para Gestion de Proyectos de Inteligencia Artificial.

La solucion simula una cartera ficticia de clientes hipotecarios y expone una API para estimar el riesgo de incumplimiento. No utiliza datos reales ni informacion interna.

## Componentes

- Generacion de datos sinteticos.
- Entrenamiento de modelo con scikit-learn.
- API con FastAPI.
- Frontend sencillo con Streamlit.
- Contenedores Docker para API y frontend.
- Pruebas automatizadas con pytest.

## Estructura

```text
bbva_mortgage_risk_project/
├── app/
│   ├── main.py
│   ├── model_utils.py
│   ├── risk_rules.py
│   └── schemas.py
├── data/
│   └── clientes_hipotecarios.csv
├── frontend/
│   └── streamlit_app.py
├── models/
│   ├── modelo_riesgo_hipotecario.pkl
│   └── modelo_metadata.json
├── tests/
│   └── test_api.py
├── train_model.py
├── requirements.txt
├── Dockerfile
├── Dockerfile.frontend
├── docker-compose.yml
└── README.md
```

## Ejecucion local

Instalar dependencias:

```bash
pip install -r requirements.txt
```

Entrenar modelo:

```bash
python train_model.py
```

Levantar API:

```bash
uvicorn app.main:app --reload
```

Abrir documentacion interactiva:

```text
http://127.0.0.1:8000/docs
```

Levantar frontend en otra terminal:

```bash
streamlit run frontend/streamlit_app.py
```

Abrir frontend:

```text
http://localhost:8501
```

## Pruebas

```bash
pytest -q
```

## Docker

Construir y ejecutar API y frontend:

```bash
docker compose up --build
```

API:

```text
http://localhost:8000/docs
```

Frontend:

```text
http://localhost:8501
```

## Endpoints principales

- GET /health
- GET /model-info
- POST /predict
- POST /predict-batch

## Ejemplo de entrada POST /predict

```json
{
  "edad": 36,
  "ingreso_mensual": 62000,
  "antiguedad_laboral_meses": 72,
  "score_buro": 735,
  "atrasos_12m": 0,
  "cuentas_abiertas": 4,
  "monto_credito": 1800000,
  "valor_vivienda": 2800000,
  "plazo_meses": 240,
  "tasa_interes_anual": 10.2,
  "deuda_mensual_actual": 4500,
  "tipo_empleo": "asalariado",
  "canal": "sucursal",
  "zona": "urbana"
}
```

## Nota academica

Este proyecto se construye con datos sinteticos para demostrar integracion entre modelo, API, frontend, pruebas, Docker y documentacion tecnica. No debe utilizarse para decisiones reales de credito.
'''.strip() + "\n"

(PROJECT_DIR / "frontend" / "streamlit_app.py").write_text(frontend_py, encoding="utf-8")
(PROJECT_DIR / "tests" / "test_api.py").write_text(test_api_py, encoding="utf-8")
(PROJECT_DIR / "requirements.txt").write_text(requirements_txt, encoding="utf-8")
(PROJECT_DIR / "Dockerfile").write_text(dockerfile_api, encoding="utf-8")
(PROJECT_DIR / "Dockerfile.frontend").write_text(dockerfile_frontend, encoding="utf-8")
(PROJECT_DIR / "docker-compose.yml").write_text(docker_compose, encoding="utf-8")
(PROJECT_DIR / ".dockerignore").write_text(dockerignore, encoding="utf-8")
(PROJECT_DIR / ".gitignore").write_text(gitignore, encoding="utf-8")
(PROJECT_DIR / "README.md").write_text(readme_md, encoding="utf-8")

print("Frontend, pruebas, Docker y README creados correctamente.")

Frontend, pruebas, Docker y README creados correctamente.


In [17]:
import subprocess
import sys
from pathlib import Path

PROJECT_DIR = Path.home() / "bbva_mortgage_risk_project"

print("Entrenando modelo...")
train_result = subprocess.run(
    [sys.executable, "train_model.py"],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True
)

print(train_result.stdout)

if train_result.returncode != 0:
    print(train_result.stderr)
    raise RuntimeError("Error durante el entrenamiento del modelo.")

print("Ejecutando pruebas...")
test_result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q"],
    cwd=PROJECT_DIR,
    capture_output=True,
    text=True
)

print(test_result.stdout)

if test_result.returncode != 0:
    print(test_result.stderr)
    raise RuntimeError("Fallaron las pruebas.")

print("Proyecto generado, entrenado y validado correctamente.")

Entrenando modelo...
Modelo entrenado correctamente.
{
  "accuracy": 0.806,
  "precision": 0.6389,
  "recall": 0.7561,
  "f1": 0.6926,
  "roc_auc": 0.8706,
  "default_rate_dataset": 0.2891,
  "confusion_matrix": {
    "tn": 1175,
    "fp": 247,
    "fn": 141,
    "tp": 437
  }
}

Ejecutando pruebas...
....                                                                     [100%]
============================== warnings summary ===============================
..\AppData\Roaming\Python\Python312\site-packages\fastapi\testclient.py:1
  C:\Users\MI42678\AppData\Roaming\Python\Python312\site-packages\fastapi\testclient.py:1: StarletteDeprecationWarning: Using `httpx` with `starlette.testclient` is deprecated; install `httpx2` instead.
    from starlette.testclient import TestClient as TestClient  # noqa

-- Docs: https://docs.pytest.org/en/stable/how-to/capture-warnings.html
4 passed, 1 warning in 6.46s

Proyecto generado, entrenado y validado correctamente.


In [18]:
import sys
import json
from pathlib import Path

PROJECT_DIR = Path.home() / "bbva_mortgage_risk_project"

if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from app.model_utils import predecir_riesgo

cliente_prueba = {
    "edad": 36,
    "ingreso_mensual": 62000,
    "antiguedad_laboral_meses": 72,
    "score_buro": 735,
    "atrasos_12m": 0,
    "cuentas_abiertas": 4,
    "monto_credito": 1800000,
    "valor_vivienda": 2800000,
    "plazo_meses": 240,
    "tasa_interes_anual": 10.2,
    "deuda_mensual_actual": 4500,
    "tipo_empleo": "asalariado",
    "canal": "sucursal",
    "zona": "urbana"
}

resultado = predecir_riesgo(cliente_prueba)

print(json.dumps(resultado, indent=2, ensure_ascii=False))

{
  "probabilidad_incumplimiento": 0.0957,
  "nivel_riesgo": "Riesgo bajo",
  "decision_sugerida": "Preaprobar sujeto a validacion documental.",
  "pago_mensual_estimado": 17609.58,
  "ltv": 0.6429,
  "ratio_deuda_ingreso": 0.3566,
  "factores_riesgo": [
    "No se identificaron alertas criticas con las reglas explicativas."
  ],
  "factores_mitigantes": [
    "LTV saludable: existe mayor aportacion inicial o menor exposicion relativa.",
    "Score de buro alto.",
    "Sin atrasos registrados en los ultimos 12 meses.",
    "Antiguedad laboral estable.",
    "Ingreso asalariado: perfil con mayor estabilidad documental."
  ],
  "nota": "Resultado generado con datos ficticios para un proyecto academico. No usar para decisiones reales de credito."
}


In [20]:
import sys
import os
import time
import socket
import subprocess
from pathlib import Path

PROJECT_DIR = Path.home() / "bbva_mortgage_risk_project"
LOG_DIR = PROJECT_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)

def puerto_abierto(host="127.0.0.1", port=8000, timeout=1):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

if puerto_abierto(port=8000):
    print("La API ya está corriendo en http://127.0.0.1:8000")
else:
    api_log = open(LOG_DIR / "api.log", "w", encoding="utf-8")

    api_process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "uvicorn",
            "app.main:app",
            "--host",
            "127.0.0.1",
            "--port",
            "8000"
        ],
        cwd=PROJECT_DIR,
        stdout=api_log,
        stderr=api_log,
        text=True
    )

    print("Iniciando API, espera unos segundos...")

    for _ in range(20):
        if puerto_abierto(port=8000):
            break
        time.sleep(1)

    if puerto_abierto(port=8000):
        print("API iniciada correctamente.")
        print("Abre este link:")
        print("http://127.0.0.1:8000/docs")
    else:
        print("No se pudo iniciar la API.")
        print("Revisa el log en:")
        print(LOG_DIR / "api.log")

Iniciando API, espera unos segundos...
API iniciada correctamente.
Abre este link:
http://127.0.0.1:8000/docs


In [21]:
import requests
import json

payload = {
    "edad": 36,
    "ingreso_mensual": 62000,
    "antiguedad_laboral_meses": 72,
    "score_buro": 735,
    "atrasos_12m": 0,
    "cuentas_abiertas": 4,
    "monto_credito": 1800000,
    "valor_vivienda": 2800000,
    "plazo_meses": 240,
    "tasa_interes_anual": 10.2,
    "deuda_mensual_actual": 4500,
    "tipo_empleo": "asalariado",
    "canal": "sucursal",
    "zona": "urbana"
}

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json=payload,
    timeout=10
)

print("Status code:", response.status_code)
print(json.dumps(response.json(), indent=2, ensure_ascii=False))

Status code: 200
{
  "probabilidad_incumplimiento": 0.0957,
  "nivel_riesgo": "Riesgo bajo",
  "decision_sugerida": "Preaprobar sujeto a validacion documental.",
  "pago_mensual_estimado": 17609.58,
  "ltv": 0.6429,
  "ratio_deuda_ingreso": 0.3566,
  "factores_riesgo": [
    "No se identificaron alertas criticas con las reglas explicativas."
  ],
  "factores_mitigantes": [
    "LTV saludable: existe mayor aportacion inicial o menor exposicion relativa.",
    "Score de buro alto.",
    "Sin atrasos registrados en los ultimos 12 meses.",
    "Antiguedad laboral estable.",
    "Ingreso asalariado: perfil con mayor estabilidad documental."
  ],
  "nota": "Resultado generado con datos ficticios para un proyecto academico. No usar para decisiones reales de credito."
}


In [22]:
import sys
import os
import time
import socket
import subprocess
from pathlib import Path

PROJECT_DIR = Path.home() / "bbva_mortgage_risk_project"
LOG_DIR = PROJECT_DIR / "logs"
LOG_DIR.mkdir(exist_ok=True)

def puerto_abierto(host="127.0.0.1", port=8501, timeout=1):
    try:
        with socket.create_connection((host, port), timeout=timeout):
            return True
    except OSError:
        return False

if puerto_abierto(port=8501):
    print("El frontend ya está corriendo en http://localhost:8501")
else:
    env = os.environ.copy()
    env["API_URL"] = "http://127.0.0.1:8000"

    frontend_log = open(LOG_DIR / "frontend.log", "w", encoding="utf-8")

    frontend_process = subprocess.Popen(
        [
            sys.executable,
            "-m",
            "streamlit",
            "run",
            "frontend/streamlit_app.py",
            "--server.address",
            "127.0.0.1",
            "--server.port",
            "8501",
            "--server.headless",
            "true",
            "--browser.gatherUsageStats",
            "false"
        ],
        cwd=PROJECT_DIR,
        stdout=frontend_log,
        stderr=frontend_log,
        text=True,
        env=env
    )

    print("Iniciando frontend, espera unos segundos...")

    for _ in range(25):
        if puerto_abierto(port=8501):
            break
        time.sleep(1)

    if puerto_abierto(port=8501):
        print("Frontend iniciado correctamente.")
        print("Abre este link:")
        print("http://localhost:8501")
    else:
        print("No se pudo iniciar el frontend.")
        print("Revisa el log en:")
        print(LOG_DIR / "frontend.log")

Iniciando frontend, espera unos segundos...
Frontend iniciado correctamente.
Abre este link:
http://localhost:8501


In [ ]:
for nombre_proceso in ["api_process", "frontend_process"]:
    proceso = globals().get(nombre_proceso)

    if proceso is not None and proceso.poll() is None:
        proceso.terminate()
        print(f"{nombre_proceso} detenido.")
    else:
        print(f"{nombre_proceso} no estaba activo o ya se había detenido.")